### Pipeline

<pre>
Capture frame
    ↓
Convert BGR → RGB
    ↓
Detect faces
    ↓
Extract embeddings
    ↓
Match against database
    ↓
Draw boxes + labels
    ↓
Show frame
    ↓
  Repeat
</pre>

### Step 1 — Capturing Frame and Converting it to PIL RGB

- Cv2 captures BRG

- Both PIL and MTCNN expect RGB

So we must convert.

Why This Matters

If you skip conversion:

- Faces may not detect properly
- Embeddings may be wrong

Because red and blue channels are swapped.

### Imports

In [1]:
import torch
import torch.nn.functional as F
import cv2
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import os

### Device and Models

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(
    image_size=160,
    margin=0,
    device=device,
    keep_all=True   # allow multiple faces
)

resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

### Database

In [3]:
def build_database(folder_path):

    database = {}

    for person in os.listdir(folder_path):

        person_path = os.path.join(folder_path, person)

        if not os.path.isdir(person_path): # True if the path is a folder.
            continue

        for file in os.listdir(person_path):

            img_path = os.path.join(person_path, file)

            img = Image.open(img_path)

            face = mtcnn(img)

            if face is None:
                continue

            if face.ndim == 3:
                face = face.unsqueeze(0)

            face = face.to(device)

            with torch.no_grad():
                embedding = resnet(face)

            embedding = embedding / embedding.norm(dim=1, keepdim=True)

            database[person] = embedding
            break  # take first valid image only (base version)

    return database

#### `face = mtcnn(img)`

MTCNN does:

1. Detect face

1. Crop it

1. Align it (eyes horizontal)

1. Resize to 160x160

1. Convert to tensor

Output shape:

`3 x 160 x 160`

OR if keep_all=True:

`N x 3 x 160 x 160`

#### `face.unsqueeze(0)`

If shape is:

`3 x 160 x 160`

Neural network expects batch dimension:

`1 x 3 x 160 x 160`

`unsqueeze(0)` adds batch dimension.

Think of it like:

> Wrap image inside a list.

### `resnet(face)`

This passes image through neural network.

Input:

`1 x 3 x 160 x 160`

Output:

`1 x 512`

Meaning: <br>
512-dimensional embedding.

#### `embedding.norm(dim=1, keepdim=True)`

Computes vector length.

If embedding = [a1, a2, ..., a512]

Norm =

`sqrt(a1² + a2² + ... + a512²)`

#### `embedding / embedding.norm(...)`

This makes vector length = 1.

Why?

Because cosine similarity works properly only with unit vectors.

### Recognition Function

In [4]:
def recognize_faces(frame, database, threshold=0.6):

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)

    boxes, _ = mtcnn.detect(img)

    if boxes is None:
        return frame

    faces = mtcnn(img)

    if faces is None:
        return frame

    if faces.ndim == 3:
        faces = faces.unsqueeze(0)

    faces = faces.to(device)

    with torch.no_grad():
        embeddings = resnet(faces)

    # Normalize embeddings
    embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

    for i, embedding in enumerate(embeddings):

        embedding = embedding.unsqueeze(0)

        best_match = None
        max_similarity = -1

        for name, db_embedding in database.items():

            similarity = F.cosine_similarity(
                embedding,
                db_embedding
            ).item()

            if similarity > max_similarity:
                max_similarity = similarity
                best_match = name

        if max_similarity > threshold:
            label = f"{best_match} ({max_similarity:.2f})"
            color = (0, 255, 0)
        else:
            label = "Unknown"
            color = (0, 0, 255)

        x1, y1, x2, y2 = map(int, boxes[i])
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            frame,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

    return frame

#### `F.cosine_similarity()`

Formula:

`cos(theta) = (A · B) / (||A|| ||B||)`

But since vectors are normalized:

    ||A|| = 1
    ||B|| = 1

So it simplifies to:

`A · B`

Range:

- 1 → identical

- 0 → unrelated

- -1 → opposite

In face recognition:

- Same person ≈ 0.7–0.9

- Different person ≈ 0.2–0.5

#### `cv2.rectangle()`

Draws bounding box.

Arguments:

`cv2.rectangle(image, (x1,y1), (x2,y2), color, thickness)`

Color format = BGR (not RGB).

#### `cv2.putText()`

Draws text on frame.

Arguments:

- image

- text

- position

- font

- size

- color

- thickness

### Main Webcam Loop

In [5]:
database = build_database("faces/single")

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = recognize_faces(frame, database)

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()